In [1]:
q1 = "I just discovered the course. Can I still join it?"
q2 = "I just found out about the program. Can I still enroll?"

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
v1 = model.encode(q1)

In [4]:
v1.shape

(384,)

In [5]:
v2 = model.encode(q2)

In [6]:
v2.shape

(384,)

In [7]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [8]:
v1.dot(dv)

np.float32(0.38779128)

In [9]:
v2.dot(dv)

np.float32(0.461237)

In [10]:
import sys
from pathlib import Path

AGENTS_DIR = Path.cwd().parent / "agents"
sys.path.append(str(AGENTS_DIR))

from ingest import load_faq_data

documents = load_faq_data()

In [11]:
documents[10]

{'id': '2b5ff70c77',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Do I need to enroll in the course before submitting homework?',
 'answer': 'No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.'}

In [12]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [13]:
texts[10]

'Do I need to enroll in the course before submitting homework? No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.'

In [14]:
from tqdm.auto import tqdm

In [15]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/30 [00:00<?, ?it/s]

1456

Hand by hand vector search

In [16]:
import numpy as np
X = np.array(vectors)

In [17]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [32]:
scores = X.dot(v_query)

In [33]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(645), np.float32(0.7629412))

In [34]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [35]:
top5 = np.argsort(scores)[-5:]

In [28]:
top5 = top5[::-1]
top5

array([ 650,  482,   29, 1209,  645])

In [36]:
scores

array([0.23396148, 0.05057266, 0.00561463, ..., 0.20289949, 0.1374665 ,
       0.09721465], shape=(1456,), dtype=float32)

In [37]:
scores[top5]

array([0.56009996, 0.65363115, 0.7192131 , 0.75793713, 0.7629412 ],
      dtype=float32)

In [38]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.56009996
{'id': '068529125b', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.'}

0.65363115
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start dat

Minsearch vector search

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [ ]:
results[0]

Filtrar por curso

In [ ]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

RAG Vector Search with Gemini

In [ ]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

instructions = """
Your task is to answer questions from the course participants
based on the provided context.
If the answer is not found in the context, respond with "I don't know."
""".strip()

load_dotenv()
genai_client = genai.Client()

def build_prompt(question, search_results):
    context = "\n\n".join(
        f"section: {d['section']}\nquestion: {d['question']}\nanswer: {d['text']}"
        for d in search_results
    )
    return f"Question:\n{question}\n\nContext:\n{context}"

def llm(prompt, model="gemini-3.8-flash"):
    response = genai_client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(system_instruction=instructions),
    )
    return response.text